In [1]:
import sys


In [2]:
import sys

print(sys.executable)
print(sys.version)

c:\Users\shafi\number-simplex-reproduction\.venv\Scripts\python.exe
3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]


In [3]:
from pathlib import Path

raw_dir = Path("../data/raw")

for subject_dir in sorted(raw_dir.iterdir()):
    if subject_dir.is_dir():
        print(f"\nSubject: {subject_dir.name}")

        for path in sorted(subject_dir.rglob("*")):
            if path.is_file():
                print("   ", path.relative_to(subject_dir))


Subject: YFF
    .DS_Store
    arithmetic\photoBehavEvents.mat
    arithmetic\spikes.mat
    dot\photoBehavEvents.mat

Subject: YFI
    .DS_Store
    arithmetic\photoBehavEvents.mat
    arithmetic\spikes.mat
    dot\photoBehavEvents.mat
    dot\spikes.mat

Subject: YFJ
    .DS_Store
    arithmetic\photoBehavEvents.mat
    arithmetic\spikes.mat
    dot\photoBehavEvents.mat
    dot\spikes.mat

Subject: YFK
    .DS_Store
    arithmetic\photoBehavEvents.mat
    arithmetic\spikesArithmetic.mat
    dot\photoBehavEvents.mat
    dot\spikesDot.mat

Subject: YFL
    .DS_Store
    arithmetic\photoBehavEvents.mat
    arithmetic\spikesArithmetic.mat
    dot\photoBehavEvents.mat
    dot\spikesDot.mat

Subject: YFM
    .DS_Store
    arithmetic\photoBehavEvents.mat
    arithmetic\spikesArithmetic.mat
    dot\photoBehavEvents.mat
    dot\spikesDot.mat

Subject: YFP
    .DS_Store
    arithmetic\photoBehavEvents.mat
    arithmetic\spikes.mat

Subject: YFR
    .DS_Store
    arithmetic\photoBehavEvents.ma

In [4]:
import h5py

file_path = "../data/raw/YFF/arithmetic/spikes.mat"

with h5py.File(file_path, "r") as f:
    print("Top-level keys:")
    for key in f.keys():
        print(key)
        

Top-level keys:
#refs#
regionsVect
spikes


In [8]:
import h5py

file_path = "../data/raw/YFF/arithmetic/spikes.mat"

with h5py.File(file_path, "r") as f:
    print("Top-level keys:")
    for key in f.keys():
        print(key)

Top-level keys:
#refs#
regionsVect
spikes


In [9]:
import h5py

file_path = "../data/raw/YFF/arithmetic/spikes.mat"

with h5py.File(file_path, "r") as f:
    for key in ["regionsVect", "spikes"]:
        obj = f[key]

        print(f"\n{key}")
        print("Type :", type(obj))
        print("Shape:", obj.shape)
        print("Dtype:", obj.dtype)


regionsVect
Type : <class 'h5py._hl.dataset.Dataset'>
Shape: (1, 54)
Dtype: object

spikes
Type : <class 'h5py._hl.group.Group'>


AttributeError: 'Group' object has no attribute 'shape'

In [10]:
import h5py

file_path = "../data/raw/YFF/arithmetic/spikes.mat"

with h5py.File(file_path, "r") as f:
    spikes_group = f["spikes"]

    print("Objects inside spikes:")
    
    for key in spikes_group.keys():
        obj = spikes_group[key]
        print(
            key,
            "| type:", type(obj),
            "| shape:", getattr(obj, "shape", None),
            "| dtype:", getattr(obj, "dtype", None)
        )
        

Objects inside spikes:
data | type: <class 'h5py._hl.dataset.Dataset'> | shape: (501537,) | dtype: uint8
ir | type: <class 'h5py._hl.dataset.Dataset'> | shape: (501537,) | dtype: uint64
jc | type: <class 'h5py._hl.dataset.Dataset'> | shape: (708277,) | dtype: uint64


In [11]:
import h5py

file_path = "../data/raw/YFF/arithmetic/spikes.mat"

with h5py.File(file_path, "r") as f:

    spikes = f["spikes"]

    print("SPIKES GROUP ATTRIBUTES")
    for key, value in spikes.attrs.items():
        print(key, ":", value)

    print("\nDATA ATTRIBUTES")
    for key, value in spikes["data"].attrs.items():
        print(key, ":", value)

    print("\nIR ATTRIBUTES")
    for key, value in spikes["ir"].attrs.items():
        print(key, ":", value)

    print("\nJC ATTRIBUTES")
    for key, value in spikes["jc"].attrs.items():
        print(key, ":", value)

SPIKES GROUP ATTRIBUTES
MATLAB_class : b'logical'
MATLAB_int_decode : 1
MATLAB_sparse : 54

DATA ATTRIBUTES

IR ATTRIBUTES

JC ATTRIBUTES


In [12]:
import h5py
import numpy as np
from scipy.sparse import csc_matrix

file_path = "../data/raw/YFF/arithmetic/spikes.mat"

with h5py.File(file_path, "r") as f:
    spikes = f["spikes"]

    data = spikes["data"][:]
    ir = spikes["ir"][:]
    jc = spikes["jc"][:]

    n_rows = int(spikes.attrs["MATLAB_sparse"])
    n_cols = len(jc) - 1

spike_matrix = csc_matrix(
    (data, ir, jc),
    shape=(n_rows, n_cols)
)

print("Shape:", spike_matrix.shape)
print("Nonzero entries:", spike_matrix.nnz)
print("Data values:", np.unique(spike_matrix.data))

Shape: (54, 708276)
Nonzero entries: 501537
Data values: [1]


In [13]:
import h5py

file_path = "../data/raw/YFF/arithmetic/spikes.mat"

with h5py.File(file_path, "r") as f:
    regions = f["regionsVect"]

    print("Shape:", regions.shape)
    print("Dtype:", regions.dtype)

    refs = regions[0]

    for i, ref in enumerate(refs[:10]):
        obj = f[ref]
        print(i, obj[()])

Shape: (1, 54)
Dtype: object
0 [[97]
 [99]
 [99]]
1 [[97]
 [99]
 [99]]
2 [[97]
 [99]
 [99]]
3 [[97]
 [99]
 [99]]
4 [[97]
 [99]
 [99]]
5 [[97]
 [99]
 [99]]
6 [[97]
 [99]
 [99]]
7 [[97]
 [99]
 [99]]
8 [[101]
 [110]
 [116]]
9 [[104]
 [112]
 [ 99]]


In [14]:
import h5py

file_path = "../data/raw/YFF/arithmetic/spikes.mat"

with h5py.File(file_path, "r") as f:
    regions = f["regionsVect"]

    region_labels = []

    for ref in regions[0]:
        obj = f[ref]
        char_codes = obj[()].flatten()

        label = "".join(chr(int(c)) for c in char_codes)
        region_labels.append(label)

for i, label in enumerate(region_labels):
    print(f"Neuron {i:2d}: {label}")

Neuron  0: acc
Neuron  1: acc
Neuron  2: acc
Neuron  3: acc
Neuron  4: acc
Neuron  5: acc
Neuron  6: acc
Neuron  7: acc
Neuron  8: ent
Neuron  9: hpc
Neuron 10: hpc
Neuron 11: hpc
Neuron 12: hpc
Neuron 13: hpc
Neuron 14: hpc
Neuron 15: hpc
Neuron 16: hpc
Neuron 17: acc
Neuron 18: acc
Neuron 19: acc
Neuron 20: acc
Neuron 21: acc
Neuron 22: acc
Neuron 23: acc
Neuron 24: acc
Neuron 25: acc
Neuron 26: hpc
Neuron 27: hpc
Neuron 28: hpc
Neuron 29: hpc
Neuron 30: hpc
Neuron 31: hpc
Neuron 32: hpc
Neuron 33: hpc
Neuron 34: hpc
Neuron 35: hpc
Neuron 36: hpc
Neuron 37: hpc
Neuron 38: hpc
Neuron 39: hpc
Neuron 40: hpc
Neuron 41: hpc
Neuron 42: hpc
Neuron 43: hpc
Neuron 44: hpc
Neuron 45: hpc
Neuron 46: hpc
Neuron 47: hpc
Neuron 48: hpc
Neuron 49: hpc
Neuron 50: hpc
Neuron 51: hpc
Neuron 52: hpc
Neuron 53: hpc


In [16]:
from collections import Counter

region_counts = Counter(region_labels)

print("Region counts:")
for region, count in sorted(region_counts.items()):
    print(f"{region}: {count}")

Region counts:
acc: 17
ent: 1
hpc: 36


In [17]:
from collections import Counter

region_counts = Counter(region_labels)

print("Region counts:")
for region, count in sorted(region_counts.items()):
    print(f"{region}: {count}")

Region counts:
acc: 17
ent: 1
hpc: 36


In [18]:
import h5py

file_path = "../data/raw/YFF/arithmetic/spikes.mat"

with h5py.File(file_path, "r") as f:
    print("FILE ATTRIBUTES:")
    for key, value in f.attrs.items():
        print(key, ":", value)

    print("\nSPIKES ATTRIBUTES:")
    for key, value in f["spikes"].attrs.items():
        print(key, ":", value)

    print("\nREGIONSVECT ATTRIBUTES:")
    for key, value in f["regionsVect"].attrs.items():
        print(key, ":", value)

FILE ATTRIBUTES:

SPIKES ATTRIBUTES:
MATLAB_class : b'logical'
MATLAB_int_decode : 1
MATLAB_sparse : 54

REGIONSVECT ATTRIBUTES:
MATLAB_class : b'cell'


In [20]:
import h5py

behav_path = "../data/raw/YFF/arithmetic/photoBehavEvents.mat"

with h5py.File(behav_path, "r") as f:
    print("Top-level keys:")

    for key in f.keys():
        obj = f[key]

        print(
            key,
            "| type:", type(obj),
            "| shape:", getattr(obj, "shape", None),
            "| dtype:", getattr(obj, "dtype", None)
        )

OSError: Unable to synchronously open file (file signature not found)

In [21]:
from scipy.io import loadmat

behav_path = "../data/raw/YFF/arithmetic/photoBehavEvents.mat"

behav = loadmat(behav_path)

print("Top-level keys:")
for key in behav.keys():
    print(key, type(behav[key]), getattr(behav[key], "shape", None))

Top-level keys:
__header__ <class 'bytes'> None
__version__ <class 'str'> None
__globals__ <class 'list'> None
photoBehavEvents <class 'scipy.io.matlab._mio5_params.MatlabOpaque'> (1,)
__function_workspace__ <class 'numpy.ndarray'> (1, 23024)


In [22]:
writetable(photoBehavEvents, 'photoBehavEvents.csv')

NameError: name 'writetable' is not defined

In [ ]:
import pandas as pd

behav = pd.read_csv("../data/raw/YFF/arithmetic/photoBehavEvents.csv")

behav.head()